In [1]:
# Cell 1 — Imports & config

import os
from pathlib import Path
import pandas as pd

PROJECTS_ROOT = Path("/Volumes/LaCie/Astrophotography/Projects")

TIF_EXTENSIONS = {".tif", ".tiff"}

# Directory names that mark calibration/stacking folders — don't descend past these.
STOP_DIR_NAMES = {"bias", "dark", "flat", "light", "stack"}

# Composite filenames to ignore (case-insensitive) — not the object's own final image.
IGNORE_TIF_NAMES = {"nebula.tif", "nebulaf.tif", "galaxies.tif", "galaxiesf.tif", "galaxy.tif", "galaxyf.tif", "stars.tif"}

print(f"Projects root: {PROJECTS_ROOT} (exists: {PROJECTS_ROOT.exists()})")


Projects root: /Volumes/LaCie/Astrophotography/Projects (exists: True)


In [2]:
# Cell 2 — Discovery

def find_candidate_tiffs(object_folder: Path) -> list[Path]:
    """Recursively collect candidate TIFFs under object_folder, pruning calibration/stack subfolders."""
    candidates = []
    for root, dirs, files in os.walk(object_folder):
        dirs[:] = [d for d in dirs if d.lower() not in STOP_DIR_NAMES]
        for filename in files:
            if filename.startswith("."):
                continue  # skip AppleDouble metadata files (e.g. "._m16.tif")
            path = Path(root) / filename
            if path.suffix.lower() not in TIF_EXTENSIONS:
                continue
            if filename.lower() in IGNORE_TIF_NAMES:
                continue
            candidates.append(path)
    return sorted(candidates)


def discover_dso_candidates() -> dict[str, list[Path]]:
    """DSO/<FolderName>/ → objectId = folder name lowercased."""
    result = {}
    dso_root = PROJECTS_ROOT / "DSO"
    if not dso_root.exists():
        print(f"  WARNING: {dso_root} not found")
        return result
    for folder in sorted(dso_root.iterdir()):
        if not folder.is_dir():
            continue
        result[folder.name.lower()] = find_candidate_tiffs(folder)
    return result


In [3]:
# Cell 3 — Run discovery & build review table

candidates = discover_dso_candidates()

rows = []
for object_id, paths in candidates.items():
    if not paths:
        rows.append({"objectId": object_id, "candidate": None, "size_mb": None})
    for path in paths:
        rows.append({
            "objectId": object_id,
            "candidate": str(path.relative_to(PROJECTS_ROOT)),
            "size_mb": round(path.stat().st_size / (1024 * 1024), 1),
        })

candidates_df = pd.DataFrame(rows, columns=["objectId", "candidate", "size_mb"])

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 100)
candidates_df


,objectId,candidate,size_mb
0,barnard72,DSO/Barnard72/barnard72.tif,66.8
1,ic1805,DSO/IC1805/c8/IC18052.tif,66.9
2,ic1805,DSO/IC1805/redcat/IC1805.tif,148.6
3,ic410,DSO/IC410/IC410-HOO.tif,62.0
4,ic410,DSO/IC410/IC410.tif,62.0
5,ic410,DSO/IC410/tadpoles.tif,66.9
6,ic434,DSO/IC434/1/IC434.tif,65.1
7,ic434,DSO/IC434/2/IC434.tif,44.9
8,ic434,DSO/IC434/3/IC434.tif,117.0
9,ic443,DSO/IC443/IC443.tif,148.8


In [4]:
# Cell 4 — Flag objects with zero or multiple candidates for manual review

counts = candidates_df.groupby("objectId")["candidate"].apply(lambda s: s.notna().sum())
needs_review = counts[counts != 1]
print(f"{len(counts)} DSO objects scanned; {len(needs_review)} need review (0 or multiple candidates):")
needs_review


48 DSO objects scanned; 7 need review (0 or multiple candidates):


objectId
ic1805     2
ic410      3
ic434      3
m16        2
m17        3
m51        2
ngc2264    2
Name: candidate, dtype: int64

In [5]:
# Cell 5 — Output directory

OUTPUT_DIR = Path("/Users/jwatts/Documents/astrophotography/Backup")  # set this to a local folder on the laptop before running

if not str(OUTPUT_DIR):
    raise ValueError("Set OUTPUT_DIR to a local folder path before running this cell")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output dir: {OUTPUT_DIR}")

Output dir: /Users/jwatts/Documents/astrophotography/Backup


In [6]:
# Cell 6 — Copy candidates to OUTPUT_DIR

import shutil

def dest_filename(object_id: str, path: Path, object_folder: Path, num_candidates: int) -> str:
    ext = path.suffix.lower()
    if num_candidates == 1:
        return f"{object_id}{ext}"
    subdir_parts = path.relative_to(object_folder).parts[:-1]
    suffix = "-".join(subdir_parts) if subdir_parts else path.stem
    return f"{object_id}-{suffix}{ext}"

copied = []
for object_id, paths in candidates.items():
    if not paths:
        continue
    object_folder = PROJECTS_ROOT / "DSO" / object_id
    # folder names on disk may not be lowercase; find the actual folder to compute relative paths
    for folder in (PROJECTS_ROOT / "DSO").iterdir():
        if folder.is_dir() and folder.name.lower() == object_id:
            object_folder = folder
            break
    for path in paths:
        dest = OUTPUT_DIR / dest_filename(object_id, path, object_folder, len(paths))
        shutil.copy2(path, dest)
        copied.append({"objectId": object_id, "source": str(path.relative_to(PROJECTS_ROOT)), "dest": dest.name})

print(f"Copied {len(copied)} files to {OUTPUT_DIR}")
pd.DataFrame(copied, columns=["objectId", "source", "dest"])

Copied 58 files to /Users/jwatts/Documents/astrophotography/Backup


,objectId,source,dest
0,barnard72,DSO/Barnard72/barnard72.tif,barnard72.tif
1,ic1805,DSO/IC1805/c8/IC18052.tif,ic1805-c8.tif
2,ic1805,DSO/IC1805/redcat/IC1805.tif,ic1805-redcat.tif
3,ic410,DSO/IC410/IC410-HOO.tif,ic410-IC410-HOO.tif
4,ic410,DSO/IC410/IC410.tif,ic410-IC410.tif
5,ic410,DSO/IC410/tadpoles.tif,ic410-tadpoles.tif
6,ic434,DSO/IC434/1/IC434.tif,ic434-1.tif
7,ic434,DSO/IC434/2/IC434.tif,ic434-2.tif
8,ic434,DSO/IC434/3/IC434.tif,ic434-3.tif
9,ic443,DSO/IC443/IC443.tif,ic443.tif
